# Importing libraries

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

from datasets import load_dataset

from sklearn.metrics import precision_score, recall_score, f1_score
from memory_profiler import memory_usage
from time import perf_counter

# Importing dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

train_df = ds['train'].to_pandas()
val_df = ds['validation'].to_pandas()
test_df = ds['test'].to_pandas()

train_df

,text,label
0,the rock is destined to be the 21st century's ...,1
1,"the gorgeously elaborate continuation of "" the...",1
2,effective but too-tepid biopic,1
3,if you sometimes like to go to the movies to h...,1
4,"emerges as something rare , an issue movie tha...",1
...,...,...
8525,any enjoyment will be hinge from a personal th...,0
8526,if legendary shlockmeister ed wood had ever ma...,0
8527,hardly a nuanced portrait of a young woman's b...,0
8528,"interminably bleak , to say nothing of boring .",0


# Dataset preprocessing

In [3]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.tolist()
        self.labels = labels.tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=False,  # Make sure this is False
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [4]:
MAX_LEN = 128
BATCH_SIZE = 32

# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Create datasets
train_dataset = TextDataset(
    texts=train_df['text'],
    labels=train_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

val_dataset = TextDataset(
    texts=val_df['text'],
    labels=val_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

test_dataset = TextDataset(
    texts=test_df['text'],
    labels=test_df['label'],
    tokenizer=tokenizer,
    max_len=MAX_LEN
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# Neural network class (LSTM)

In [5]:
class LSTMClassifier(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super(LSTMClassifier, self).__init__()

        self.embedding = nn.Embedding(tokenizer.vocab_size, embedding_dim)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids):
        # 1. Embedding lookup
        embedded = self.embedding(input_ids)

        # 2. LSTM forward pass
        outputs, (hidden, cell) = self.lstm(embedded)

        # 3. Extract the final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        else:
            hidden = hidden[-1,:,:]

        # 4. Apply dropout
        hidden = self.dropout(hidden)  

        # 5. Final classification layer (returns logits)
        output = self.fc(hidden)

        return output

# Instancing the LSTM model, criterion and optimizer

In [6]:
embedding_dim = 128
hidden_dim = 128
output_dim = 1
n_layers = 2
bidirectional = True
dropout = 0.3

model = LSTMClassifier(
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    output_dim=output_dim,
    n_layers=n_layers,
    bidirectional=bidirectional,
    dropout=dropout
)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using {device} device')
model = model.to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

Using cuda device


# Training and evaluation functions

In [8]:
def train_epoch(model, data_loader, optimizer, criterion, device):
    """
    One epoch of training. 
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels' in each batch.
    - optimizer, criterion: training components (e.g., Adam, BCEWithLogitsLoss).
    - device: 'cpu' or 'cuda'.
    """
    model.train()
    losses = []
    correct_predictions = 0

    # For calculating precision, recall, F1:
    all_labels = []
    all_preds = []

    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 1) Forward pass -> raw logits
        logits = model(input_ids)  # shape: (batch_size, 1)
        logits = logits.squeeze(dim=1)  # shape: (batch_size,)

        # 2) Compute loss (BCEWithLogitsLoss expects raw logits)
        loss = criterion(logits, labels.float())

        # 3) Backprop + optimization
        loss.backward()
        optimizer.step()

        # 4) Track loss
        losses.append(loss.item())

        # 5) Convert logits -> probabilities -> predicted classes
        probs = torch.sigmoid(logits)          # in [0, 1]
        preds_cls = (probs >= 0.5).long()      # threshold at 0.5

        # 6) Count correct predictions
        correct_predictions += torch.sum(preds_cls == labels)

        # 7) Collect for metric calculation
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds_cls.cpu().numpy())

    # Calculate overall metrics for the epoch
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1


def eval_model(model, data_loader, criterion, device):
    """
    Evaluation function. Similar to train_epoch, but no backprop.
    - model: your LSTMClassifier (returns raw logits).
    - data_loader: iterable with 'input_ids' and 'labels'.
    - criterion: e.g., BCEWithLogitsLoss for binary classification.
    - device: 'cpu' or 'cuda'.
    """
    model.eval()
    losses = []
    correct_predictions = 0

    all_labels = []
    all_preds = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            # 1) Forward pass -> logits
            logits = model(input_ids)  # shape: (batch_size, 1)
            logits = logits.squeeze(dim=1)  # shape: (batch_size,)

            # 2) Compute loss
            loss = criterion(logits, labels.float())
            losses.append(loss.item())

            # 3) Convert logits -> probabilities -> predicted classes
            probs = torch.sigmoid(logits)
            preds_cls = (probs >= 0.5).long()

            correct_predictions += torch.sum(preds_cls == labels)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds_cls.cpu().numpy())

    # Metrics
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    accuracy = float(correct_predictions) / len(data_loader.dataset)
    avg_loss = sum(losses) / len(losses)

    return accuracy, avg_loss, precision, recall, f1

# Training loop

In [9]:
def training_loop(epochs):
    for epoch in range(epochs):
        print(f'Epoch {epoch + 1}/{epochs}')
        
        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_loader, optimizer, criterion, device)
        
        val_acc, val_loss, val_prec, val_rec, val_f1 = eval_model(
            model, val_loader, criterion, device)
        
        print(f'Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}, '
            f'Precision: {train_prec:.4f}, Recall: {train_rec:.4f}, F1 Score: {train_f1:.4f}')
        
        print(f'Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}, '
            f'Precision: {val_prec:.4f}, Recall: {val_rec:.4f}, F1 Score: {val_f1:.4f}')
    return train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1

In [ ]:
seeds = [2,3,5]
EPOCHS = 5
results = pd.DataFrame(columns=['seed', 'train_loss', 'train_acc', 'train_prec', 'train_rec', 'train_f1',
                                'val_loss', 'val_acc', 'val_prec', 'val_rec', 'val_f1',
                                'test_loss', 'test_acc', 'test_prec', 'test_rec', 'test_f1',
                                'max_memory_usage_train', 'max_vram_usage_train', 'total_time_train',
                                'max_memory_usage_test', 'max_vram_usage_test', 'total_time_test'])

for seed in seeds:
    torch.manual_seed(seed)
    # Resetting model
    model = LSTMClassifier(
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        n_layers=n_layers,
        bidirectional=bidirectional,
        dropout=dropout
    )
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss().to(device)
    
    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_train = perf_counter()
    max_memory_usage_train, retval = memory_usage((training_loop, (EPOCHS,), {}), retval=True, max_usage=True)
    total_time_train = perf_counter() - start_time_train

    max_vram_usage_train = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    train_acc, train_loss, train_prec, train_rec, train_f1, val_acc, val_loss, val_prec, val_rec, val_f1 = retval

    if torch.cuda.is_available():
        torch.cuda.reset_max_memory_allocated()

    start_time_test = perf_counter()
    max_memory_usage_test, retval = memory_usage((eval_model, (model, test_loader, criterion, device), {}), retval=True, max_usage=True)
    total_time_test = perf_counter() - start_time_test

    max_vram_usage_test = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else None

    test_acc, test_loss, test_prec, test_rec, test_f1 = retval

    results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
                                                val_loss, val_acc, val_prec, val_rec, val_f1,
                                                test_loss, test_acc, test_prec, test_rec, test_f1,
                                                max_memory_usage_train, max_vram_usage_train, total_time_train,
                                                max_memory_usage_test, max_vram_usage_test, total_time_test]],
                                                columns=results.columns)], ignore_index=True)
    

c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6645, Accuracy: 0.5899, Precision: 0.5877, Recall: 0.6026, F1 Score: 0.5950
Val Loss: 0.6173, Accuracy: 0.6614, Precision: 0.6581, Recall: 0.6717, F1 Score: 0.6648
Epoch 2/5
Train Loss: 0.5502, Accuracy: 0.7304, Precision: 0.7376, Recall: 0.7151, F1 Score: 0.7262
Val Loss: 0.5686, Accuracy: 0.7073, Precision: 0.7346, Recall: 0.6492, F1 Score: 0.6892
Epoch 3/5
Train Loss: 0.4162, Accuracy: 0.8175, Precision: 0.8244, Recall: 0.8068, F1 Score: 0.8155
Val Loss: 0.6160, Accuracy: 0.7195, Precision: 0.7120, Recall: 0.7373, F1 Score: 0.7244
Epoch 4/5
Train Loss: 0.2947, Accuracy: 0.8848, Precision: 0.8941, Recall: 0.8729, F1 Score: 0.8834
Val Loss: 0.6850, Accuracy: 0.7139, Precision: 0.6906, Recall: 0.7749, F1 Score: 0.7303
Epoch 5/5
Train Loss: 0.2063, Accuracy: 0.9263, Precision: 0.9331, Recall: 0.9184, F1 Score: 0.9257
Val Loss: 0.8612, Accuracy: 0.7223, Precision: 0.7484, Recall: 0.6698, F1 Score: 0.7069


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
C:\Users\Rafael\AppData\Local\Temp\ipykernel_25284\3438238464.py:46: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([[seed, train_loss, train_acc, train_prec, train_rec, train_f1,
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6715, Accuracy: 0.5782, Precision: 0.5894, Recall: 0.5156, F1 Score: 0.5500
Val Loss: 0.6349, Accuracy: 0.6473, Precision: 0.6536, Recall: 0.6266, F1 Score: 0.6398
Epoch 2/5
Train Loss: 0.5621, Accuracy: 0.7163, Precision: 0.7358, Recall: 0.6750, F1 Score: 0.7041
Val Loss: 0.5980, Accuracy: 0.6970, Precision: 0.7075, Recall: 0.6717, F1 Score: 0.6891
Epoch 3/5
Train Loss: 0.4287, Accuracy: 0.8110, Precision: 0.8242, Recall: 0.7906, F1 Score: 0.8071
Val Loss: 0.5849, Accuracy: 0.7036, Precision: 0.7256, Recall: 0.6548, F1 Score: 0.6884
Epoch 4/5
Train Loss: 0.3105, Accuracy: 0.8748, Precision: 0.8894, Recall: 0.8560, F1 Score: 0.8724
Val Loss: 0.6876, Accuracy: 0.7036, Precision: 0.7602, Recall: 0.5947, F1 Score: 0.6674
Epoch 5/5
Train Loss: 0.2125, Accuracy: 0.9234, Precision: 0.9327, Recall: 0.9128, F1 Score: 0.9226
Val Loss: 0.8049, Accuracy: 0.7167, Precision: 0.7192, Recall: 0.7111, F1 Score: 0.7151


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


Epoch 1/5
Train Loss: 0.6610, Accuracy: 0.6022, Precision: 0.6092, Recall: 0.5705, F1 Score: 0.5892
Val Loss: 0.6315, Accuracy: 0.6520, Precision: 0.7132, Recall: 0.5084, F1 Score: 0.5936
Epoch 2/5
Train Loss: 0.5418, Accuracy: 0.7312, Precision: 0.7342, Recall: 0.7247, F1 Score: 0.7294
Val Loss: 0.5760, Accuracy: 0.7054, Precision: 0.7078, Recall: 0.6998, F1 Score: 0.7038
Epoch 3/5
Train Loss: 0.4109, Accuracy: 0.8211, Precision: 0.8277, Recall: 0.8110, F1 Score: 0.8193
Val Loss: 0.5818, Accuracy: 0.7261, Precision: 0.7261, Recall: 0.7261, F1 Score: 0.7261
Epoch 4/5
Train Loss: 0.2905, Accuracy: 0.8846, Precision: 0.8950, Recall: 0.8715, F1 Score: 0.8831
Val Loss: 0.6136, Accuracy: 0.7205, Precision: 0.7571, Recall: 0.6492, F1 Score: 0.6990
Epoch 5/5
Train Loss: 0.1976, Accuracy: 0.9260, Precision: 0.9353, Recall: 0.9154, F1 Score: 0.9252
Val Loss: 0.7198, Accuracy: 0.7355, Precision: 0.7535, Recall: 0.6998, F1 Score: 0.7257


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\torch\cuda\memory.py:343: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(


In [11]:
results.to_csv('results/lstm_binary2.csv', index=False)
results.head()

,seed,train_loss,train_acc,train_prec,train_rec,train_f1,val_loss,val_acc,val_prec,val_rec,...,test_acc,test_prec,test_rec,test_f1,max_memory_usage_train,max_vram_usage_train,total_time_train,max_memory_usage_test,max_vram_usage_test,total_time_test
0,2,0.206283,0.926260,0.933063,0.918406,0.925676,0.861192,0.722326,0.748428,0.669794,...,0.751407,0.779167,0.701689,0.738401,1239.582031,231.628906,21.833574,1239.605469,194.781250,0.776991
1,3,0.212515,0.923447,0.932678,0.912778,0.922621,0.804935,0.716698,0.719165,0.711069,...,0.760788,0.759328,0.763602,0.761459,1240.062500,232.615234,23.666420,1240.062500,195.835938,0.780962
2,5,0.197616,0.926026,0.935314,0.915358,0.925228,0.719776,0.735460,0.753535,0.699812,...,0.739212,0.745665,0.726079,0.735741,1254.972656,231.113281,23.757436,1254.929688,194.156250,0.777920


In [12]:
torch.save(model.state_dict(), 'results/lstm_binary2.pth')